<div align="center">
<span style="font-size: 2.5em;">Xenon S1S2 Model Performance Readout</span>
<br/>
<span style="font-size: 1.2em; color: gray;">Comprehensive evaluation across signal-only and signal+background configurations</span>
</div>

## Overview

This notebook reads out Xenon S1S2 model checkpoints.

- **Signal Configurations**: Signal-only vs Signal+Background (mu=150 ER/kg/day)
- **Binning**: Fixed to 10 bins (training setup)

**Metrics Evaluated**:
- Best validation loss
- Best validation accuracy

## Configuration

In [1]:
# =============================
# IMPORTS AND PATH SETTINGS
# =============================

import os
import sys
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tabulate import tabulate

# Navigate to root
ROOT_NAME = "xenon-sbi"
while os.path.basename(os.getcwd()) != ROOT_NAME:
    os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
# =============================
# USER CONFIGURATION
# =============================

# SPECIFY THE HALO MODEL HERE
halo = "shm"  # Options: "lmc", "shm", "shmpp", "combined"

# Model parameters
n_train = 300_000

# Training was done with fixed binning
bins = 10

# Background settings
mu_bg = 150  # ER events per kg per day

print(f"\n{'='*60}")
print(f"HALO MODEL: {halo.upper()}")
if halo == "combined":
    print(f"Training samples: {3*n_train:,}")
else:
    print(f"Training samples: {n_train:,}")
print(f"Bins: {bins}")
print(f"Background (ERs): {mu_bg}")
print(f"{'='*60}")


HALO MODEL: SHM
Training samples: 300,000
Bins: 10
Background (ERs): 150


## Signal-Only Models

In [3]:
# =============================
# SIGNAL-ONLY
# =============================

print(f"\n{'='*80}")
print("SIGNAL-ONLY (mu_bg = 0)")
print(f"{'='*80}\n")

signal_type = "signal_only"
modelname = "S1S2_signal"

signal_only_result = None

from configs.config import load_model_s1s2

MODELPATH = f"models/xenon/{signal_type}/{halo}/{modelname}_bins{bins}_n{n_train}_{halo}.pt"

if not os.path.exists(MODELPATH):
    print(f"  Model not found: {MODELPATH}")
else:
    try:
        model, ckpt = load_model_s1s2(
            MODELPATH,
            bins=bins,
            modelname=modelname,
            device="cpu",
            print_arch=False
        )

        signal_only_result = {
            "label": "Signal-only",
            "path": MODELPATH,
            "model": model,
            "checkpoint": ckpt,
            "best_val_loss": ckpt.get("best_val_loss", np.nan),
            "best_val_acc": ckpt.get("best_val_acc", np.nan),
        }

        print("  Loaded")
        print(f"    - Best Val Loss: {signal_only_result['best_val_loss']:.4f}")
        print(f"    - Best Val Acc:  {signal_only_result['best_val_acc']:.4f}")
    except Exception as e:
        print(f"  Error loading model: {e}")

print(f"\nLoaded {1 if signal_only_result is not None else 0} signal-only model")


SIGNAL-ONLY (mu_bg = 0)

  Loaded
    - Best Val Loss: 0.4194
    - Best Val Acc:  0.7710

Loaded 1 signal-only model


## Signal + Background Models

In [4]:
# =============================
# SIGNAL+BACKGROUND
# =============================

print(f"\n{'='*80}")
print(f"SIGNAL + BACKGROUND (mu_bg = {mu_bg})")
print(f"{'='*80}\n")

signal_type = f"signal_bg_mu{mu_bg:.0f}"
modelname = "S1S2_signal_bg"

signal_bg_result = None

MODELPATH = f"models/xenon/{signal_type}/{halo}/{modelname}_bins{bins}_n{n_train}_{halo}.pt"

if not os.path.exists(MODELPATH):
    print(f"  Model not found: {MODELPATH}")
else:
    try:
        model, ckpt = load_model_s1s2(
            MODELPATH,
            bins=bins,
            modelname=modelname,
            device="cpu",
            print_arch=False
        )

        signal_bg_result = {
            "label": f"Signal+Background (mu={mu_bg})",
            "path": MODELPATH,
            "model": model,
            "checkpoint": ckpt,
            "best_val_loss": ckpt.get("best_val_loss", np.nan),
            "best_val_acc": ckpt.get("best_val_acc", np.nan),
        }

        print("  Loaded")
        print(f"    - Best Val Loss: {signal_bg_result['best_val_loss']:.4f}")
        print(f"    - Best Val Acc:  {signal_bg_result['best_val_acc']:.4f}")
    except Exception as e:
        print(f"  Error loading model: {e}")

print(f"\nLoaded {1 if signal_bg_result is not None else 0} signal+background model")


SIGNAL + BACKGROUND (mu_bg = 150)

  Loaded
    - Best Val Loss: 0.5086
    - Best Val Acc:  0.6994

Loaded 1 signal+background model


## Performance Summary

In [5]:
# =============================
# SUMMARY TABLE
# =============================

results = [
    signal_only_result,
    signal_bg_result,
]

rows = []
for r in results:
    if r is None:
        continue
    rows.append([
        r["label"],
        f"{r['best_val_loss']:.4f}",
        f"{r['best_val_acc']:.4f}",
    ])

print(f"\nMODEL PERFORMANCE SUMMARY (HALO={halo.upper()}, BINS={bins})")
print("="*72)
if rows:
    print(tabulate(
        rows,
        headers=["Model", "Best Val Loss", "Best Val Acc"],
        tablefmt="grid"
    ))
else:
    print("No models loaded.")


MODEL PERFORMANCE SUMMARY (HALO=SHM, BINS=10)
+----------------------------+-----------------+----------------+
| Model                      |   Best Val Loss |   Best Val Acc |
+============================+=================+================+
| Signal-only                |          0.4194 |         0.771  |
+----------------------------+-----------------+----------------+
| Signal+Background (mu=150) |          0.5086 |         0.6994 |
+----------------------------+-----------------+----------------+


## Detailed Model Information

In [6]:
# =============================
# ACCESS MODEL OBJECTS
# =============================

print("\nModel objects are stored in the following variables:")
print("\nSignal-Only Models:")
print("  - signal_only_result")
print("\nSignal+Background Models:")
print("  - signal_bg_result")
print("\nEach entry contains:")
print("  - 'path': Model file path")
print("  - 'model': PyTorch model object")
print("  - 'checkpoint': Training checkpoint dict")
print("  - 'best_val_loss': Best validation loss")
print("  - 'best_val_acc': Best validation accuracy")
print("\nExample usage:")
print("  model = signal_only_result['model']")
print("  checkpoint = signal_only_result['checkpoint']")


Model objects are stored in the following variables:

Signal-Only Models:
  - signal_only_result

Signal+Background Models:
  - signal_bg_result

Each entry contains:
  - 'path': Model file path
  - 'model': PyTorch model object
  - 'checkpoint': Training checkpoint dict
  - 'best_val_loss': Best validation loss
  - 'best_val_acc': Best validation accuracy

Example usage:
  model = signal_only_result['model']
  checkpoint = signal_only_result['checkpoint']
